# 通过类定义工具

本 notebook 全程只围绕一个文件：`s020/s020_agent_loop_openai_toolclass.py`（下文简称 **程序 / 类版**）。
四步进阶：

1. **先跑通不带工具的 `agent_loop`** —— 看清这个循环最小形态到底在干什么；
2. **再对比「用类定义工具」vs「只用函数定义工具」** —— 优缺点各是什么；
3. **然后照着 `BaseTool` 手写一个类工具并真的用起来** —— 以 `read_file` 为例；
4. **最后用股票工具 `get_stock_price_date` 出题，并给出参考答案。**

> **全程真实调用**：大模型客户端直接用程序里的 `cls_style.client`，循环直接用程序里的 `cls_style.agent_loop`，
> 不使用任何「假 LLM」。

## 目录

1. [不带工具的 `agent_loop`](#1-不带工具的-agent_loop)
2. [类定义工具 vs 函数定义工具](#2-类定义工具-vs-函数定义工具)（纯本地验证，不发请求）
3. [用 `BaseTool` 定义类工具：`read_file` 实操](#3-用-basetool-定义类工具read_file-实操)
4. [练习题：`get_stock_price_date`（含参考答案）](#4-练习题get_stock_price_date含参考答案)
5. [避坑清单与作业](#5-避坑清单与作业)

> 提示：第 1、3、4 节会真实请求大模型，逐节运行会产生少量 token 消耗；
> 只想看工具定义本身的话，只跑第 2 节即可（它不发任何请求）。

## 1. 不带工具的 `agent_loop`

### 1.1 把程序里的三样东西借出来

`agent_loop` 的外部依赖全部是可注入的关键字参数，这正是后面「直接调用」的基础。

> 程序在 `import` 那一刻就会创建 OpenAI 客户端，所以必须**先**把 `OPENAI_API_KEY` 配好
> （真 key 放在 `.env` 里，`load_dotenv` 会自动向上查找），再 import。

In [23]:
import inspect
import json
import os
import sys
from pathlib import Path

S020 = Path(r"e:\2609\智能体开发\s020")     # 换成你自己的 s020 目录即可
sys.path.insert(0, str(S020))
os.chdir(S020)                             # 让相对路径与脚本内的工作目录对齐

from dotenv import load_dotenv
load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("未找到 OPENAI_API_KEY —— 本文全部走真实 API，请先在 .env 中配置再运行。")

# 类版：一个类 = 一个工具
import s020_agent_loop_openai_toolclass as cls_style
from s020_agent_loop_openai_toolclass import BaseTool, ToolRegistry, RunBashTool

# 直接借用程序里的三样东西
CLIENT = cls_style.client            # 已配好的大模型客户端（读 .env 的 OPENAI_API_KEY / OPENAI_BASE_URL）
MODEL = cls_style.MODEL              # 模型名
AGENT_LOOP = cls_style.agent_loop    # 核心循环，本文一行都不重写

print("cwd        :", Path.cwd())
print("model      :", MODEL)
print("client     :", CLIENT.base_url)
print("默认工具    :", list(cls_style.TOOL_MAP), "（程序自己注册的那个 run_bash）")
print("\nagent_loop 签名：")
print(" ", AGENT_LOOP.__module__ + ".agent_loop" + str(inspect.signature(AGENT_LOOP)))

cwd        : e:\2609\智能体开发\s020
model      : deepseek-chat
client     : https://api.deepseek.com
默认工具    : ['run_bash'] （程序自己注册的那个 run_bash）

agent_loop 签名：
  s020_agent_loop_openai_toolclass.agent_loop(messages: list, *, client=None, registry: s020_agent_loop_openai_toolclass.ToolRegistry = None, tools: list = None, model: str = None, max_rounds: int = None, max_tokens: int = 8000, verbose: bool = True) -> str


In [24]:
# 系统提示词直接用程序自带模板；无工具模式下补一句，避免模型去「假装调用」不存在的 bash
history = [
    {"role": "system", "content": cls_style.build_system()
                        + "\n本轮没有挂载任何工具，不要声称自己能执行命令或读文件。"},
    {"role": "user", "content": "你好，请介绍一下你自己。"},
]

# 不带工具：tools=[] 传进去，第一轮就会命中 return
print("第 1 轮 ->", AGENT_LOOP(history, tools=[], max_tokens=300))

第 1 轮 -> 你好！我是本次对话中的编程协助助手，可以在当前工作环境（`e:\2609\智能体开发\s020`，Windows 环境）里帮你完成编程任务。

需要说明的是：本轮没有挂载任何工具，所以我目前不能执行命令、读写文件或运行程序。也就是说，这一轮我只能和你交流，没法实际动手操作。

等工具挂载后，我可以帮你做的事情包括：

- **阅读和修改代码**：查看项目结构、定位问题、改代码、补测试
- **运行与调试**：执行构建、跑测试、复现报错并排查原因
- **脚本与自动化**：写批处理/PowerShell 脚本、做数据处理、整理文件
- **解释与设计**：讲解代码逻辑、评估方案、给实现建议

你可以直接告诉我你想做什么，比如“看一下这个项目的结构”“帮我修一个报错”“写个脚本批量重命名文件”。如果需要我实际操作，请先挂载工具，我再动手。


## 2. 类定义工具 vs 函数定义工具

这一节**不发任何请求**，全部是本地验证：定义工具、看 schema、看注册表里装的是什么。

### 2.1 先用「函数」写一个工具（基线）

一个工具 = 一个模块级函数；注册表靠**手写 dict**，schema 靠**逐行调用** `generate_tool_description`。

下面这个 `read_file` 也是第 3 节要改造的对象，先把它跑通。

In [25]:
from tool_description_generator import generate_tool_description

def read_file(path: str, offset: int = 1, limit: int = None) -> str:
    """读取文件内容，可用 offset/limit 分页，避免大文件撑爆上下文

    :param path: 文件路径
    :param offset: 从第几行开始读，从 1 开始计数
    :param limit: 本次最多读多少行，不传则读到文件末尾
    :return: 文件内容；还有剩余行时给出下一次的 offset 提示
    """
    lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    start = max(1, offset)
    end = len(lines) if not limit else min(len(lines), start + limit - 1)
    out = "\n".join(lines[start - 1:end])
    if end < len(lines):
        out += f"\n... ({len(lines) - end} more lines, continue with offset={end + 1})"
    return out


# 函数式的注册表：两份数据都要自己维护，加一个工具就要动两处
TOOL_MAP_FUNC = {"read_file": read_file}                     # 名字 -> 函数
TOOLS_FUNC = [generate_tool_description(read_file)]           # 名字 -> schema

# 造一个 12 行的演示文件
DEMO = S020 / "s020_nb_demo"
DEMO.mkdir(exist_ok=True)
(DEMO / "notes.txt").write_text(
    "\n".join(f"第 {i} 行：这是演示用文本 line {i}" for i in range(1, 13)), encoding="utf-8"
)

print("=== 函数式：读前 3 行 ===")
print(read_file("s020_nb_demo/notes.txt", limit=3))
print("\n=== 函数式：从 offset=10 读到末尾 ===")
print(read_file("s020_nb_demo/notes.txt", offset=10))
print("\nTOOL_MAP_FUNC :", TOOL_MAP_FUNC)

=== 函数式：读前 3 行 ===
第 1 行：这是演示用文本 line 1
第 2 行：这是演示用文本 line 2
第 3 行：这是演示用文本 line 3
... (9 more lines, continue with offset=4)

=== 函数式：从 offset=10 读到末尾 ===
第 10 行：这是演示用文本 line 10
第 11 行：这是演示用文本 line 11
第 12 行：这是演示用文本 line 12

TOOL_MAP_FUNC : {'read_file': <function read_file at 0x0000024FA72171A0>}


### 2.2 用「类」写同一个工具

程序里的写法：**一个 `BaseTool` 子类 = 一个工具**。子类只负责三件事 —— 给 `name`、在 `__init__` 里存配置、
把真正的逻辑写进 `execute()`（写清类型注解和 docstring 即可）。

```python
class BaseTool:                                    # ① 父类统一 schema 出口
    name: str = ""            # 必填
    description: str = ""     # 可选覆盖
    parameters: dict = {}     # 可选覆盖

    def execute(self, **kwargs) -> str: raise NotImplementedError

    def to_openai_schema(self):
        spec = generate_tool_description(self.execute)               # 解析 execute 的签名 + docstring
        spec["function"]["name"] = self.name or type(self).__name__  # ← 关键一行
        if self.description: spec["function"]["description"] = self.description
        if self.parameters:  spec["function"]["parameters"]  = self.parameters
        return spec

class RunBashTool(BaseTool):                       # ② 一个类 = 一个工具
    name = "run_bash"                              # 工具名必须写在类属性上

    def __init__(self, workdir=None, timeout=120): # 配置挂 self
        self.workdir = workdir or os.getcwd()
        self.timeout = timeout
        self.dangerous = ["rm -rf /", "sudo", "shutdown", ...]

    def execute(self, command: str) -> str: ...    # 唯一入口，写清注解 + docstring 即可

REGISTRY.register(RunBashTool(os.getcwd()))        # ③ 注册的是「工具实例」
```

配套的 `ToolRegistry` 只做四件事：`register` / `has` / `execute` / `get_definitions`；
循环里的分发也就从 `TOOL_MAP[名字](**参数)` 换成 `REGISTRY.execute(名字, 参数)`。

### 2.3 一句话区别

> 函数式是「**一个函数 = 一个工具**」，靠手写 dict 登记；
> 类式是「**一个类 = 一个工具**」，靠 `register(实例)` 登记，schema 由工具自己生成。

### 2.4 对照实验①：注册表里装的到底是什么

函数式装的是**函数对象**，类式装的是**工具实例**。这个差别决定了后面所有的优缺点。

In [26]:
print("— 函数式 TOOL_MAP_FUNC：名字 -> 函数 —")
for name, fn in TOOL_MAP_FUNC.items():
    print(f"  {name:16s} -> {type(fn).__name__:12s} __name__={fn.__name__}")

print("\n— 类式 TOOL_MAP：名字 -> 工具实例 —")
for name, tool in cls_style.TOOL_MAP.items():
    print(f"  {name:16s} -> {type(tool).__name__:12s} execute.__name__={tool.execute.__name__}")

— 函数式 TOOL_MAP_FUNC：名字 -> 函数 —
  read_file        -> function     __name__=read_file

— 类式 TOOL_MAP：名字 -> 工具实例 —
  run_bash         -> RunBashTool  execute.__name__=execute


### 2.5 对照实验②：工具名从哪来 —— `__name__` 陷阱

这是类式最容易踩的坑：

* 函数式的函数名就是工具名（`read_file`），天然可用；
* 类式所有工具的方法都叫 `execute`，`__name__` **恒为 `execute`**，
  所以 `BaseTool` 里必须有 `spec["function"]["name"] = self.name` 这一行，
  否则你注册 5 个工具，大模型看到的是 5 个都叫 `execute` 的函数。

In [27]:
print("函数式 read_file.__name__             =", read_file.__name__)
print("类式   RunBashTool().execute.__name__ =", RunBashTool().execute.__name__)
print("类式   RunBashTool.name               =", RunBashTool.name)


class NoNameTool(BaseTool):
    """反例：忘了写 name"""

    def execute(self, x: str) -> str:
        """忘了写 name 的工具
        :param x: 一个参数
        :return: 结果
        """
        return x


print("\n忘记写 name 时 schema 里的函数名 =", NoNameTool().to_openai_schema()["function"]["name"])
try:
    ToolRegistry().register(NoNameTool())
except ValueError as e:
    print("注册报错 ->", e)

函数式 read_file.__name__             = read_file
类式   RunBashTool().execute.__name__ = execute
类式   RunBashTool.name               = run_bash

忘记写 name 时 schema 里的函数名 = NoNameTool
注册报错 -> Tool class NoNameTool 缺少 name


### 2.6 对照实验③：发给大模型的 schema 完全一样

两版都用同一个 `generate_tool_description`，从 **类型注解 + docstring 的 `:param:`** 生成描述。
所以大模型看到的 JSON 结构是一致的 —— 区别只在「**谁负责生成、什么时候生成**」。

In [28]:
print("函数式 read_file schema:")
print(json.dumps(TOOLS_FUNC[0], ensure_ascii=False, indent=2))
print("\n类式 run_bash schema:")
print(json.dumps(cls_style.TOOLS[0], ensure_ascii=False, indent=2))

函数式 read_file schema:
{
  "type": "function",
  "function": {
    "name": "read_file",
    "description": "读取文件内容，可用 offset/limit 分页，避免大文件撑爆上下文",
    "parameters": {
      "type": "object",
      "properties": {
        "path": {
          "type": "string",
          "description": "文件路径"
        },
        "offset": {
          "type": "integer",
          "description": "从第几行开始读，从 1 开始计数"
        },
        "limit": {
          "type": "integer",
          "description": "本次最多读多少行，不传则读到文件末尾"
        }
      },
      "required": [
        "path"
      ]
    }
  }
}

类式 run_bash schema:
{
  "type": "function",
  "function": {
    "name": "run_bash",
    "description": "执行一条 shell 命令并返回它的输出",
    "parameters": {
      "type": "object",
      "properties": {
        "command": {
          "type": "string",
          "description": "要执行的 cmd 命令，例如 dir /b *.py、type a.txt、echo hi > a.txt"
        }
      },
      "required": [
        "command"
      ]
    }
  }
}


### 2.7 类式的 6 个实在优点（用实例说话）

| # | 优点 | 说明 |
| --- | --- | --- |
| ① | **配置挂在 `self` 上** | 同一份类代码按不同参数实例化 = 不同工具（`timeout` / `max_lines`） |
| ② | **依赖可注入** | 数据源 / DB session 从构造函数传入，测试时换成 mock，不用改模块级全局 |
| ③ | **统一入口 + 统一兜底** | 只有 `execute` 是入口；`ToolRegistry.execute()` 内部 `try/except`，工具炸了也返回 `Error: ...` 文本喂回模型，让模型自我纠正 |
| ④ | **可继承、可复用** | 子类改签名即可改 schema；公共父类可统一加日志、计时、权限 |
| ⑤ | **helper 天然不暴露** | 类式只有 `execute` 会进注册表，类里其他方法（如 `_helper`）都不会被当成工具；函数式要靠命名约定 + 手写 dict 自觉维护 |
| ⑥ | **扩展只动一处** | 加工具 = 加一个类 + 一行 `register`，`agent_loop` 和 `TOOLS` 常量都不用改 |

In [29]:
# ① 同一份类代码，不同实例 = 不同配置（函数式只能改全局或用 functools.partial）
strict = RunBashTool(workdir=str(S020), timeout=5)
loose = RunBashTool(workdir=str(S020), timeout=300)
print("① strict.timeout =", strict.timeout, "| loose.timeout =", loose.timeout)

# ③ 危险命令被拦、未知工具、工具内部报错 —— 全部转成 Error 文本回给模型
print("\n③ 黑名单  :", strict.execute(command="sudo rm -rf /"))
print("  未知工具 :", cls_style.REGISTRY.execute("no_such_tool", {}))
print("  照常执行 :", strict.execute(command="echo hello from class tool"))


# ⑥ 加一个工具 = 加一个类 + 一行 register，循环与常量都不用动
class EchoTool(BaseTool):
    """把输入原样返回，用来观察 Registry 的分发"""

    name = "echo_back"

    def execute(self, text: str) -> str:
        """把输入原样返回

        :param text: 任意文本
        :return: 原样返回该文本
        """
        return text


tmp_registry = ToolRegistry().register(RunBashTool(str(S020))).register(EchoTool())
print("\n⑥ 新注册表 :", list(tmp_registry.as_map()))
print("  分发测试  :", tmp_registry.execute("echo_back", {"text": "加一个工具真的只要两行"}))
print("  tools     :", [t["function"]["name"] for t in tmp_registry.get_definitions()])

① strict.timeout = 5 | loose.timeout = 300

③ 黑名单  : Error: Dangerous command blocked
  未知工具 : Error: Unknown tool no_such_tool
  照常执行 : hello from class tool

⑥ 新注册表 : ['run_bash', 'echo_back']
  分发测试  : 加一个工具真的只要两行
  tools     : ['run_bash', 'echo_back']


### 2.8 类式的 5 个代价（客观看，别神化）

| # | 代价 | 说明 |
| --- | --- | --- |
| ① | **样板代码变多** | 一个 5 行的函数要写成「类 + `name` + `__init__` + `execute`」十几行；只有 1~2 个参数的小工具，函数式明显更划算 |
| ② | **`name` 必填且易忘** | 所有工具的 `execute.__name__` 都叫 `execute`，忘了写 `name` 就退化成类名，注册直接报错（见 2.5） |
| ③ | **类数量膨胀** | 20 个工具 = 20 个类；函数式就是 20 个函数，文件更紧凑 |
| ④ | **多一层间接** | 调试时多经过 `to_openai_schema()` 和 `registry.execute()`，调用栈变深 |
| ⑤ | **共享状态要另想办法** | 一组工具共用一份配置 / 一个 DB 连接时，函数式用模块级全局就行；类式每个类各存一份，要么逐个传参，要么再包一层 |

另外还有一个**类属性的可变默认值坑**：`description` / `parameters` 在 `BaseTool` 里是**类属性**，
如果直接在实例上改子键，会污染到所有实例。下面用代码看清楚。

In [ ]:
class SharedParamsTool(BaseTool):
    """反例：把可变的 schema 覆盖写成了类属性"""

    name = "shared_params"
    parameters = {}          # ← 类属性：所有实例共享同一个 dict

    def execute(self, x: str) -> str:
        """演示用
        :param x: 一个参数
        :return: 原样返回
        """
        return x


a, b = SharedParamsTool(), SharedParamsTool()
a.parameters["properties"] = {"x": {"type": "string"}}     # 只想改 a
print("⑤ 只改了 a，b 也跟着变？", "properties" in b.parameters)   # True —— 踩坑


class SafeParamsTool(BaseTool):
    """正解：可变配置放进 __init__，变成实例属性"""

    name = "safe_params"

    def __init__(self):
        self.parameters = {}                              # ← 实例属性，互不干扰

    def execute(self, x: str) -> str:
        """演示用
        :param x: 一个参数
        :return: 原样返回
        """
        return x


a2, b2 = SafeParamsTool(), SafeParamsTool()
a2.parameters["properties"] = {"x": {"type": "string"}}
print("   改了 a2，b2 受影响？", "properties" in b2.parameters)   # False

### 2.9 对比小结

| 维度 | 函数式（模块级函数） | **类式（`BaseTool` 子类）** |
| --- | --- | --- |
| 一个工具 = | 一个函数 | 一个 `BaseTool` 子类 |
| 工具名来源 | 函数名 `__name__` | 类属性 `name`（**必填**） |
| schema 生成 | 手工逐个调 `generate_tool_description` | 工具自己 `to_openai_schema()` |
| 配置 / 状态 | 模块级全局、闭包、`functools.partial` | 每个工具各自 `self` |
| 多实例不同配置 | 麻烦 | **天然支持** |
| 依赖注入 / 单测 | 要 monkey-patch 模块全局 | **构造函数注入（最干净）** |
| 私有 helper | 靠命名约定自觉维护 | **天然不暴露** |
| 异常兜底 | 调用处自己 `try/except` | **`Registry.execute()` 统一兜** |
| 继承 / 复用 | 无 | **好** |
| 代码量 | 最少 | 最多 |
| 新增一个工具 | 加函数 + 改 dict 两处 | 加一个类 + 一行 `register` |
| 更适合 | 临时脚本、两三把小工具 | 工具要独立配置、独立复用、独立单测、可继承 |

**选型口诀**：

* 只是两三行的小脚本 → 别过度设计，函数式最省事；
* 工具要独立配置 / 注入 / 单测 / 继承，或者数量会持续增长 → 一个类一个工具。

## 3. 用 `BaseTool` 定义类工具：`read_file` 实操

改造对象就是 2.1 里那个函数式 `read_file`。它本身没有配置要存，
所以类版本也很简单 —— 只有 `max_lines` 这一个可选配置，放进 `__init__` 挂到 `self` 上即可。

### 3.1 改造四步法

| 步骤 | 动作 |
| --- | --- |
| ① | 建类继承 `BaseTool`，写 `name = "read_file"`（类式必填） |
| ② | 有配置要存就搬进 `__init__` 挂到 `self`（这里是 `max_lines`）；没有就不写 |
| ③ | 把函数体原样搬进 `execute()`，签名照抄 |
| ④ | **docstring 与 `:param:` 一个字都不要改** —— 它就是 schema 的唯一来源 |

In [30]:
class ReadFileTool(BaseTool):
    """读取文件内容（支持 offset/limit 分页），一个类 = 一个工具"""

    name = "read_file"                       # ① 工具名：类式必须显式给

    def __init__(self, max_lines=2000):
        # ② 配置挂在 self 上：一份类可以实例化成多份不同上限的工具
        self.max_lines = max_lines

    def execute(self, path: str, offset: int = 1, limit: int = None) -> str:
        """读取文件内容，可用 offset/limit 分页，避免大文件撑爆上下文

        :param path: 文件路径
        :param offset: 从第几行开始读，从 1 开始计数
        :param limit: 本次最多读多少行，不传则读到文件末尾
        :return: 文件内容；还有剩余行时给出下一次的 offset 提示
        """                                   # ④ docstring 与 :param: 一个字不改
        target = Path(path)
        if not target.exists():
            return f"Error: no such file: {path}"   # 出错返回文本，别抛异常

        lines = target.read_text(encoding="utf-8", errors="replace").splitlines()
        start = max(1, offset)
        room = min(limit or self.max_lines, self.max_lines)   # ② 上限由实例配置决定
        end = len(lines) if not limit else min(len(lines), start + room - 1)
        out = "\n".join(lines[start - 1:end])
        if end < len(lines):
            out += f"\n... ({len(lines) - end} more lines, continue with offset={end + 1})"
        return out

### 3.2 本地验证：直接调 `execute()`

不经过大模型也能单独验证，这是类式工具好测的直接体现。

In [31]:
reader = ReadFileTool()                        # 实例化

print("=== 读前 3 行 ===")
print(reader.execute(path="s020_nb_demo/notes.txt", limit=3))

print("\n=== 从 offset=10 读到末尾 ===")
print(reader.execute(path="s020_nb_demo/notes.txt", offset=10))

print("\n=== 文件不存在（返回 Error 文本，不抛异常）===")
print(reader.execute(path="no_such_file.txt"))

print("\n=== 同一份类，换个 max_lines 就是另一把尺 ===")
small = ReadFileTool(max_lines=2)
print(small.execute(path="s020_nb_demo/notes.txt", limit=99))   # limit=99 被 max_lines=2 压住

=== 读前 3 行 ===
第 1 行：这是演示用文本 line 1
第 2 行：这是演示用文本 line 2
第 3 行：这是演示用文本 line 3
... (9 more lines, continue with offset=4)

=== 从 offset=10 读到末尾 ===
第 10 行：这是演示用文本 line 10
第 11 行：这是演示用文本 line 11
第 12 行：这是演示用文本 line 12

=== 文件不存在（返回 Error 文本，不抛异常）===
Error: no such file: no_such_file.txt

=== 同一份类，换个 max_lines 就是另一把尺 ===
第 1 行：这是演示用文本 line 1
第 2 行：这是演示用文本 line 2
... (10 more lines, continue with offset=3)


### 3.3 自动生成 schema

`to_openai_schema()` 内部就是 `generate_tool_description(self.execute)`，
所以**描述来自 docstring 第一行，参数说明来自 `:param:`，是否必填来自有没有默认值**。

这也解释了为什么第 ④ 步要求「docstring 一个字都不要改」—— 它就是给大模型看的说明书。

In [32]:
schema = reader.to_openai_schema()
print(json.dumps(schema, ensure_ascii=False, indent=2))

params = schema["function"]["parameters"]
print("\nrequired =", params["required"], "（只有没给默认值的 path）")

{
  "type": "function",
  "function": {
    "name": "read_file",
    "description": "读取文件内容，可用 offset/limit 分页，避免大文件撑爆上下文",
    "parameters": {
      "type": "object",
      "properties": {
        "path": {
          "type": "string",
          "description": "文件路径"
        },
        "offset": {
          "type": "integer",
          "description": "从第几行开始读，从 1 开始计数"
        },
        "limit": {
          "type": "integer",
          "description": "本次最多读多少行，不传则读到文件末尾"
        }
      },
      "required": [
        "path"
      ]
    }
  }
}

required = ['path'] （只有没给默认值的 path）


### 3.4 注册：装进 `ToolRegistry`

`ToolRegistry` 是「名字 → 工具实例」的字典，同时负责生成 `tools=` 列表和分发调用。

In [34]:
REGISTRY = ToolRegistry()
REGISTRY.register(RunBashTool(workdir=str(S020)))   # 直接复用程序里现成的类
REGISTRY.register(ReadFileTool())  # 我们刚写的类

print("已注册工具 :", list(REGISTRY.as_map()))
print("has('read_file') =", REGISTRY.has("read_file"))

# 这就是要塞给 OpenAI 的 tools= 参数
print("\ntools 名称 :", [t["function"]["name"] for t in REGISTRY.get_definitions()])

已注册工具 : ['run_bash', 'read_file']
has('read_file') = True

tools 名称 : ['run_bash', 'read_file']


### 3.5 真实调用：直接调 `agent_loop`

调用方式只有一行 —— `registry` 传进去，其余全用程序里的默认值（真客户端、真模型）：

```python
AGENT_LOOP(messages, registry=REGISTRY)
```

对照第 1 节：唯一的差别就是把 `tools=[]` 换成了 `registry=REGISTRY`（有工具），
于是第 ② 步不再立刻命中，而是走第 ③ 步「执行工具 → `role="tool"` 回传 → 再来一轮」。

`verbose=True`（默认）会打印模型每轮说的话和工具输出，方便看清整个过程。

> 小提示：程序自己的打印格式专门高亮 `command` 参数，所以非 shell 工具会多打出一行空的 `$ `，属正常现象。

In [35]:
# 同样直接用程序自带模板，只在后面补一句工具使用约定
SYSTEM = cls_style.build_system() + (
    "\n涉及文件内容时先用 read_file 读，再回答；一次读不完就用 offset 继续读，不要编造内容。"
)

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "读 s020_nb_demo/notes.txt 的第 4~6 行，原样告诉我"},
]

# ↓ 直接调用程序里的循环：client / model 都用默认值，只注入 registry
answer = AGENT_LOOP(messages, registry=REGISTRY, max_rounds=8)
print("\n=== 最终回答 ===")
print(answer)

$ 
第 4 行：这是演示用文本 line 4
第 5 行：这是演示用文本 line 5
第 6 行：这是演示用文本 line 6
... (6 more lines, continue with offset=7)

=== 最终回答 ===
s020_nb_demo/notes.txt 第 4~6 行原文：

```
第 4 行：这是演示用文本 line 4
第 5 行：这是演示用文本 line 5
第 6 行：这是演示用文本 line 6
```

文件共 12 行，后面还有 6 行。


In [36]:
# 对比第 1 节的历史：这里出现了 assistant(tool_calls) 和 tool(结果) 两种新角色
for m in messages:
    extra = f"  tool_calls={[c['function']['name'] for c in m['tool_calls']]}" if m.get("tool_calls") else ""
    print(f"[{m.get('role', '?'):9s}] {m['content'][:60]}{extra}")

[system   ] You are a coding agent at e:\2609\智能体开发\s020. Current date a
[user     ] 读 s020_nb_demo/notes.txt 的第 4~6 行，原样告诉我
[assistant]   tool_calls=['read_file']
[tool     ] 第 4 行：这是演示用文本 line 4
第 5 行：这是演示用文本 line 5
第 6 行：这是演示用文本 line
[assistant] s020_nb_demo/notes.txt 第 4~6 行原文：

```
第 4 行：这是演示用文本 line 4



## 4. 练习题：`get_stock_price_date`（含参考答案）

### 4.1 题目

下面这个函数是「纯函数式」的工具写法，请把它改造成 `BaseTool` 子类，注册进 `REGISTRY`，并用 `AGENT_LOOP` 跑通一轮。

```python
def get_stock_price_date(symbol: str, date: str):
    """获取股票某一天的股票价格

    :param symbol: 股票代码，如 sz000002、sz300750
    :param date: 日期，如 2025-04-03
    :return: 股票价格
    """
    stock_data = ak.stock_zh_a_daily(symbol=symbol, start_date=date, end_date=date, adjust="qfq")
    print(stock_data)
    stock_data = stock_data.close.values[0]
    return f"{symbol} {date} 价格：${stock_data}"
```

### 4.2 改造要点（先自己想一想，再看答案）

1. **最简单就够了**：没有配置要存，`__init__` 可以不写 —— 直接 `import akshare as ak`，
   在 `execute()` 里调用即可。这是类工具最直接的形态。
2. **别在工具里 `print`**：`print` 只进人眼，不进模型上下文。要给模型看的信息必须写进返回值。
3. **不用手写 `try/except`**：`Registry.execute()` 已经兜住了异常，会转成 `Error: ...` 文本回给模型，
   模型看到后还能自己换个日期重试（想让提示更明确时再自己捕获即可）。
4. **空结果要心里有数**：非交易日或代码有误时 DataFrame 是空的，`close.values[0]` 会 `IndexError` ——
   会被上一层转成 Error 文本，不会把循环搞崩。
5. **日期从哪来**：模型不知道「今天」是几号。两个动作缺一不可 ——
   * docstring 里把格式写死：`日期，格式 YYYY-MM-DD`；
   * **系统提示词直接复用 `cls_style.build_system()`**，它自带当前日期与中文星期，
     模型才能把「昨天 / 上周五」换算成具体日期。

### 4.3 参考答案

In [37]:
import akshare as ak        # 数据源直接引入即可（已安装、可联网）


class StockPriceTool(BaseTool):
    """获取股票某一天收盘价：一个类 = 一个工具"""

    name = "get_stock_price_date"          # ① 工具名必填

    # ② 没有配置要存，__init__ 可以不写

    def execute(self, symbol: str, date: str) -> str:
        """获取股票某一天的股票价格

        :param symbol: 股票代码，如 sz000002、sz300750
        :param date: 日期，格式 YYYY-MM-DD，如 2025-04-03；「今天/昨天」请用系统提示词中的当前日期换算
        :return: 股票价格
        """
        # ③ 直接联网取数；异常不用管，Registry.execute() 会兜成 Error 文本回给模型
        stock_data = ak.stock_zh_a_daily(symbol=symbol, start_date=date, end_date=date, adjust="qfq")
        close = stock_data.close.values[0]
        return f"{symbol} {date} 价格：${close}"

### 4.4 直接调一次，看返回值与 schema

不经过大模型也能单独验证：实例化 → 调 `execute()` → 看 `to_openai_schema()`。

In [38]:
stock_tool = StockPriceTool()

print("=== 直接调用（真实行情）===")
print(stock_tool.execute(symbol="sz000002", date="2025-04-03"))

print("\n=== 自动生成的 schema ===")
print(json.dumps(stock_tool.to_openai_schema(), ensure_ascii=False, indent=2))

=== 直接调用（真实行情）===
sz000002 2025-04-03 价格：$7.14

=== 自动生成的 schema ===
{
  "type": "function",
  "function": {
    "name": "get_stock_price_date",
    "description": "获取股票某一天的股票价格",
    "parameters": {
      "type": "object",
      "properties": {
        "symbol": {
          "type": "string",
          "description": "股票代码，如 sz000002、sz300750"
        },
        "date": {
          "type": "string",
          "description": "日期，格式 YYYY-MM-DD，如 2025-04-03；「今天/昨天」请用系统提示词中的当前日期换算"
        }
      },
      "required": [
        "symbol",
        "date"
      ]
    }
  }
}


### 4.5 注册 + 用 `agent_loop` 真实跑一轮

大模型看不见你机器上的时钟，而 `cls_style.build_system()` **自带当前日期与中文星期**，
直接复用它，模型就能把「今天 / 昨天 / 上周五」换算成 `YYYY-MM-DD`。

In [39]:
# 还是直接用程序自带模板（自带当前日期），只补一句日期格式的约定
SYSTEM_STOCK = cls_style.build_system() + (
    "\n查股价时用 get_stock_price_date；日期格式必须是 YYYY-MM-DD，"
    "按上面的当前日期把「今天 / 昨天 / 上周五」换算好再传，不要凭记忆猜。"
)
print(SYSTEM_STOCK)

REGISTRY.register(stock_tool)
print("\n已注册:", list(REGISTRY.as_map()))

You are a coding agent at e:\2609\智能体开发\s020. Current date and time: 2026-10-24 20:42:19 星期六. Use the bash tool to solve tasks. Act, don't explain. The shell is Windows cmd.exe: use dir / type / echo / findstr / copy / del, never Unix commands like ls / cat / grep / wc.
查股价时用 get_stock_price_date；日期格式必须是 YYYY-MM-DD，按上面的当前日期把「今天 / 昨天 / 上周五」换算好再传，不要凭记忆猜。

已注册: ['run_bash', 'read_file', 'get_stock_price_date']


In [41]:
messages = [
    {"role": "system", "content": SYSTEM_STOCK},
    {"role": "user", "content": "万科A（sz000002）前天 的收盘价是多少？"},
]

# 同样只注入 registry，其余用程序默认值
print(AGENT_LOOP(messages, registry=REGISTRY, max_rounds=8))

$ 
sz000002 2026-10-22 价格：$3.31
万科A（sz000002）前天（2026-10-22）的收盘价为 **3.31 元**。


### 题目二

把下列工具也写成类放入 s020_agent_loop_openai_toolclass.py 中去，注意safe_path函数只是一个函数，不是工具不用改

提示词：

s020_类定义工具_vs_函数定义工具.ipynb:1-67
﻿​  重建新py文件，把这些工具也写成类放入中去，注意safe_path函数只是一个函数，不是工具不用改，参考
s020_agent_loop_openai_toolclass.py
﻿​  的模式

这个任务完成后此agent已经是一个可以做vibe coding的agent了

可以让他尝试做一个简单

In [ ]:
# ---------- s02 新增：三个文件工具 + 一个路径匹配工具 ----------
def safe_path(p: str) -> Path:
    """解析路径并确保它还在工作目录内

    :param p: 相对或绝对路径
    :return: 落在工作目录内的绝对路径
    """
    path = (WORKDIR / p).resolve()
    if not str(path).lower().startswith(str(WORKDIR).lower()):
        raise ValueError(f"Path escapes workspace: {p}")
    return path


def read_file(path: str, offset: int = 1, limit: int = None) -> str:
    """读取文件内容，可用 offset/limit 分页，避免大文件撑爆上下文

    :param path: 文件路径，相对路径基于工作目录
    :param offset: 从第几行开始读，从 1 开始计数
    :param limit: 本次最多读多少行，不传则读到文件末尾
    :return: 文件内容；还有剩余行时给出下一次的 offset 提示
    """
    lines = safe_path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    start = max(1, offset)
    end = len(lines) if not limit else min(len(lines), start + limit - 1)
    out = "\n".join(lines[start - 1:end])
    if end < len(lines):
        out += f"\n... ({len(lines) - end} more lines, continue with offset={end + 1})"
    return out


def write_file(path: str, content: str) -> str:
    """把完整内容写入文件（新建，或整体覆盖已有文件）

    :param path: 目标文件路径，父目录不存在时自动创建
    :param content: 文件的完整内容
    :return: 写入结果
    """
    file_path = safe_path(path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding="utf-8")
    return f"Wrote {len(content)} bytes to {path}"


def edit_file(path: str, old_text: str, new_text: str) -> str:
    """按上下文做精确补丁替换：给出一段原文，替换成新片段（不用正则，只改这一处）

    :param path: 要修改的文件路径
    :param old_text: 待替换的原文片段，需与文件内容逐字符一致，可跨多行
    :param new_text: 替换后的新内容，传空串表示删除这段原文
    :return: 修改结果
    """
    file_path = safe_path(path)
    text = file_path.read_text(encoding="utf-8")
    if old_text not in text:
        return f"Error: text not found in {path}"
    file_path.write_text(text.replace(old_text, new_text, 1), encoding="utf-8")
    return f"Edited {path}"


def glob_files(pattern: str) -> str:
    """按 glob 模式匹配文件名或路径，改代码前先定位文件

    :param pattern: glob 模式，例如 **/*.py、*.md、src/**/*.ts；** 递归匹配任意层目录
    :return: 匹配到的相对路径列表，无匹配时返回 (no matches)
    """
    matches = sorted(str(p.relative_to(WORKDIR)) for p in WORKDIR.glob(pattern))
    return "\n".join(matches) if matches else "(no matches)"

## 5. 避坑清单与作业

### 5.1 避坑清单

| 坑 | 现象 | 对策 |
| --- | --- | --- |
| 忘了写 `name` | `register` 抛 `ValueError: ... 缺少 name` | 类属性 `name` 是类式的必填项（见 2.5） |
| 同名重复注册 | `Duplicate tool name` | 用 `name` 做唯一标识，或分两个 Registry |
| 可变类属性 | 改一个实例的 `parameters`，其他实例跟着变 | 可变配置放进 `__init__`（见 2.8 ⑤） |
| docstring 首行不是描述 | 大模型看不懂工具干嘛的 | 第一行写一句完整的话，`:param:` 另起一行 |
| `:param x:` 写错 / 漏写 | 参数说明退化成 `x参数` | 名字要和签名逐字一致 |
| `execute` 返回非字符串 | 回传给模型时可能炸 | 统一 `return str(...)`，报错返回 `Error: ...` |
| 工具里 `print` | 模型看不到 | 要进上下文的内容必须写进返回值 |
| 让模型猜日期 / 路径 | 参数填错、幻觉 | 系统提示词注入当天日期 / 工作目录 |